In [ ]:
# QRLocator Project - Neural Network for QR Code Detection

# First, let's import all necessary libraries and modules
import os
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Check for GPU/TPU availability
device = None
xla_device = None

# Check for CUDA GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Found CUDA GPU: {torch.cuda.get_device_name(0)}")
# Check for TPU using PyTorch XLA
elif hasattr(torch, "xla") and torch.xla.is_available():
    device = torch.device("xla")
    xla_device = torch.xla
    print(f"Found TPU: {torch.xla._XLAC._xla_device_uname(xla_device)}_")
else:
    device = torch.device("cpu")
    print("No GPU or TPU found, using CPU")

# Ensure we can import from local modules
import sys
sys.path.append(".")

# Import project modules
from config import CFG
from model import QRViTDet
from dataset import build_dataloaders
from train_eval import train, evaluate, infer
from dataset_gen import generate_split, visualise_sample, make_background

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cuda":
    torch.cuda.manual_seed_all(42)

print(f"QRLocator Project Loaded - Device: {device.type}")

In [ ]:
# Configuration

# Display current configuration
print("=" * 50)
print("CONFIGURATION")
print("=" * 50)
print(f"Image Size: {CFG.img_w}x{CFG.img_h}")
print(f"Input Channels: {CFG.in_channels}")
print(f"Transformer Embed Dim: {CFG.embed_dim}")
print(f"Number of Encoder Layers: {CFG.enc_layers}")
print(f"Number of Decoder Layers: {CFG.dec_layers}")
print(f"Number of Attention Heads: {CFG.num_heads}")
print(f"Max QR codes per image: {CFG.max_qr_per_image}")
print(f"Training Batch Size: {CFG.batch_size}")
print(f"Training Epochs: {CFG.epochs}")
print(f"Learning Rate: {CFG.lr}")
print(f"Checkpoint Directory: {CFG.checkpoint_dir}")
print("=" * 50)

# Create checkpoint directory if it doesn't exist
CFG.checkpoint_dir.mkdir(parents=True, exist_ok=True)
print(f"Created checkpoint directory: {CFG.checkpoint_dir}")

In [ ]:
# Synthetic Data Generation

# Generate synthetic dataset (this may take a while)
print("Generating synthetic dataset...")
print(f"Train samples: {CFG.num_train}")
print(f"Val samples: {CFG.num_val}")
print(f"Test samples: {CFG.num_test}")

# Generate the dataset
generate_split("train", CFG.num_train, CFG.dataset_dir)
generate_split("val", CFG.num_val, CFG.dataset_dir)
generate_split("test", CFG.num_test, CFG.dataset_dir)

print("Dataset generation complete!")

# Visualize a sample from the dataset
print("Visualizing sample dataset...")
visualise_sample("dataset_preview.png", n=4)
print("Preview saved as 'dataset_preview.png'")

In [ ]:
# Load Dataset and Create DataLoaders

# Build the dataloaders for training and evaluation
train_dl, val_dl, test_dl = build_dataloaders(CFG)

print(f"Training batches: {len(train_dl)}")
print(f"Validation batches: {len(val_dl)}")
print(f"Test batches: {len(test_dl)}")

# Get a sample batch to verify data loading
sample_batch = next(iter(train_dl))
images, boxes_list, n_objs = sample_batch

print(f"Batch shape - Images: {images.shape}")
print(f"Batch shape - Boxes list length: {len(boxes_list)}")
for i, boxes in enumerate(boxes_list[:3]):
    print(f"  Sample {i}: {len(boxes)} QR codes detected")
print(f"Number of objects per image: {n_objs[:5]}")

In [ ]:
# Model Initialization

# Create the QRViTDet model
print(f"Using device: {device}")

# Initialize the model
model = QRViTDet(CFG).to(device)

# Print model summary
print(model)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Count parameters in backbone vs rest
backbone_params = sum(p.numel() for p in model.backbone_params)
non_backbone_params = sum(p.numel() for p in model.non_backbone_params)
print(f"Backbone parameters: {backbone_params:,}")
print(f"Non-backbone parameters: {non_backbone_params:,}")

In [ ]:
# Training the Model

# Start training process
print("Starting training...")
print(f"Epochs: {CFG.epochs}")
print(f"Batch size: {CFG.batch_size}")
print(f"Device: {device}")

# Call the training function
train(CFG)

print("Training complete!")

# Load best model for evaluation
best_checkpoint = "checkpoints/best.pt"
if os.path.exists(best_checkpoint):
    print(f"Loading best model from {best_checkpoint}")
else:
    print("Best checkpoint not found, using last.pt")
    best_checkpoint = "checkpoints/last.pt"

In [ ]:
# Model Evaluation

# Evaluate the trained model on test set
print("Evaluating model on test set...")
metrics = evaluate(best_checkpoint, CFG)

print("\n=== Test Set Metrics ===")
for metric, value in metrics.items():
    print(f"{metric:>12s}: {value}")

In [ ]:
# Inference on Sample Images

# Run inference on a sample image
import glob

# Find sample images (you can replace these with your own images)
sample_images = glob.glob("data/sample_images/*.png") + glob.glob("data/sample_images/*.jpg")

if not sample_images:
    # Create a sample image if none exist
    print("No sample images found. Creating a synthetic test image...")
    
    # Generate a synthetic scene and save it
    scene, labels = compose_scene(CFG.img_w, CFG.img_h, bg_dir=None)
    Image.fromarray(scene).save("test_image.png")
    sample_images = ["test_image.png"]
    
print(f"Found {len(sample_images)} sample image(s)")

for img_path in sample_images:
    print(f"\nProcessing: {img_path}")
    output_path = f"detected_{Path(img_path).name}"
    
    # Run inference
    boxes, confs = infer(img_path, best_checkpoint, output_path, CFG)
    
    print(f"Results saved to: {output_path}")

In [ ]:
# Model Interpretability

# Run interpretability analysis on a few test samples
print("Running model interpretability analysis...")

# Import interpretability functions
from interpret import run_interpretability

# Run with 2 samples (adjust as needed)
run_interpretability(best_checkpoint, num_samples=2)

print("Interpretability analysis complete!")

In [ ]:
# Cleanup and Summary

print("\n=== Training Summary ===")
print(f"Project: QRLocator - Neural Network QR Code Detector")
print(f"Dataset: Synthetic ({CFG.num_train} train, {CFG.num_val} val, {CFG.num_test} test)")
print(f"Model: QRViTDet (Vision Transformer + DETR-style detection)")
print(f"Best Model: {best_checkpoint}")
print(f"Final Metrics - mAP@0.5: {metrics['precision']:.4f}, F1: {metrics['f1']:.4f}")
print("\nNotebook execution completed successfully!")